# CRM Coursework – Grouping S&P 500 Stocks
Study period: 1 January 2021 – 1 January 2025 (end date exclusive). Benchmark: S&P 500 index (^GSPC). Price data source: yfinance.

In [1]:
# Setup: imports, folders and package versions
import sys, os
import pandas as pd, numpy as np, yfinance as yf, sklearn, scipy, matplotlib

for folder in ['data/raw', 'data/processed', 'outputs/figures']:
    os.makedirs(folder, exist_ok=True)

versions = pd.Series({
    'python': sys.version.split()[0],
    'pandas': pd.__version__,
    'numpy': np.__version__,
    'yfinance': yf.__version__,
    'scikit-learn': sklearn.__version__,
    'scipy': scipy.__version__,
    'matplotlib': matplotlib.__version__,
}, name='version')
versions.to_csv('outputs/package_versions.csv')
versions

python          3.12.0
pandas           3.1.1
numpy            2.5.5
yfinance         1.8.0
scikit-learn     1.9.1
scipy           1.18.1
matplotlib      3.11.2
Name: version, dtype: str

### 1.1 Ticker Selection
I took the S&P 500 company list from Wikipedia, using the version of the page from 20 November 2026 so the list stays the same if the notebook is re-run.

In [2]:
import requests
from io import StringIO
from datetime import datetime

headers = {"User-Agent": "Mozilla/5.0"}

# find the version of the wikipedia page from 20 Nov 2026
params = {
    "action": "query",
    "prop": "revisions",
    "titles": "List of S&P 500 companies",
    "rvlimit": 1,
    "rvstart": "2026-11-20T23:59:59Z",
    "rvdir": "older",
    "format": "json",
}
r = requests.get("https://en.wikipedia.org/w/api.php", params=params, headers=headers)
pages = r.json()["query"]["pages"]
revision_id = list(pages.values())[0]["revisions"][0]["revid"]

url = "https://en.wikipedia.org/w/index.php?title=List_of_S%26P_500_companies&oldid=" + str(revision_id)
html = requests.get(url, headers=headers).text
sp500 = pd.read_html(StringIO(html))[0]

# yfinance needs BRK-B instead of BRK.B
sp500["Ticker"] = sp500["Symbol"].str.replace(".", "-")

sp500["Date added"] = pd.to_datetime(sp500["Date added"], errors="coerce")
sp500["Added after 2021"] = sp500["Date added"] > "2021-01-01"

sp500["Wiki revision"] = revision_id
sp500["Downloaded on"] = datetime.now().strftime("%Y-%m-%d %H:%M")
sp500.to_csv("data/raw/sp500_constituents.csv", index=False)

print("Wikipedia revision:", revision_id)
print("Downloaded on:", sp500["Downloaded on"][0])
print("Number of companies:", len(sp500))
print("Added after 1 Jan 2021:", sp500["Added after 2021"].sum())
sp500.head()

Wikipedia revision: 1384572480
Downloaded on: 2026-11-23 17:47
Number of companies: 503
Added after 1 Jan 2021: 89


,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded,Ticker,Added after 2021,Wiki revision,Downloaded on
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902,MMM,False,1384572480,2026-11-23 17:47
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916,AOS,False,1384572480,2026-11-23 17:47
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888,ABT,False,1384572480,2026-11-23 17:47
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888),ABBV,False,1384572480,2026-11-23 17:47
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989,ACN,False,1384572480,2026-11-23 17:47


### Downloading Prices
I used yfinance to download daily data for every company and the S&P 500 index (^GSPC) from 1 January 2021 to 1 January 2025 (the end date isn't included).

In [3]:
tickers = sp500["Ticker"].tolist() + ["^GSPC"]

# auto_adjust=False so we get both Close and Adj Close
data = yf.download(tickers, start="2021-01-01", end="2025-01-01", auto_adjust=False, progress=False)

adj_close = data["Adj Close"]
close = data["Close"]
volume = data["Volume"]

adj_close.to_csv("data/raw/prices_adj_close.csv")
close.to_csv("data/raw/prices_close.csv")
volume.to_csv("data/raw/volume.csv")

failed = [t for t in tickers if adj_close[t].isna().all()]

print("Trading days:", len(adj_close))
print("First day:", adj_close.index[0].date())
print("Last day:", adj_close.index[-1].date())
print("Tickers downloaded:", len(tickers) - len(failed))
print("Tickers with no data:", failed)

$VYLR: Data doesn't exist for startDate = 1609477200, endDate = 1735707600


$FDXF: Data doesn't exist for startDate = 1609477200, endDate = 1735707600


$SNDK: Data doesn't exist for startDate = 1609477200, endDate = 1735707600


$HONA: Data doesn't exist for startDate = 1609477200, endDate = 1735707600


$Q: Data doesn't exist for startDate = 1609477200, endDate = 1735707600



5 Failed downloads:


['VYLR', 'FDXF', 'SNDK', 'HONA', 'Q']: Data doesn't exist for startDate = 1609477200, endDate = 1735707600


Trading days: 1005
First day: 2021-01-04
Last day: 2024-12-31
Tickers downloaded: 499
Tickers with no data: ['FDXF', 'HONA', 'Q', 'SNDK', 'VYLR']


### 1.2 Data Checks
Before calculating anything, I checked the data for duplicate dates, missing values, impossible prices, volume errors and possible unadjusted stock splits. I only kept shares with a complete history for the whole study period, so every share is compared over the same weeks. Nothing was filled in. Shares that failed a check are listed in the exclusion log with the reason.

In [4]:
shares = [t for t in sp500["Ticker"] if t in adj_close.columns]

print("Duplicate dates:", adj_close.index.duplicated().sum())
print("Prices zero or below:", (adj_close[shares] <= 0).sum().sum())
print("Negative volume:", (volume[shares] < 0).sum().sum())

log = []
for t in shares:
    if adj_close[t].isna().all():
        log.append([t, "No data in study period"])
    elif adj_close[t].isna().any() or close[t].isna().any() or volume[t].isna().any():
        first = adj_close[t].first_valid_index().date()
        log.append([t, "Incomplete history (first price " + str(first) + ")"])
    elif (adj_close[t] <= 0).any():
        log.append([t, "Price zero or below"])
    elif (volume[t] <= 0).any():
        log.append([t, "Zero or negative volume"])

exclusion_log = pd.DataFrame(log, columns=["Ticker", "Reason"])
exclusion_log.to_csv("outputs/exclusion_log.csv", index=False)

valid = [t for t in shares if t not in exclusion_log["Ticker"].values]

# look for big one-day jumps that could be unadjusted stock splits
daily_change = close[valid].pct_change()
big_moves = daily_change[daily_change.abs() > 0.5].stack().dropna()

# the benchmark only needs prices, its volume isn't used
print("^GSPC missing prices:", adj_close["^GSPC"].isna().sum())

print("Shares checked:", len(shares))
print("Shares excluded:", len(exclusion_log))
print("Shares kept:", len(valid))
print("Possible split days to review:", len(big_moves))
print(big_moves)
exclusion_log

Duplicate dates: 0
Prices zero or below: 0
Negative volume: 0


^GSPC missing prices: 0
Shares checked: 503
Shares excluded: 17
Shares kept: 486
Possible split days to review: 3
Date        Ticker
2023-06-08  CVNA      0.560206
2024-04-11  GL       -0.531402
2024-11-15  BE        0.591867
dtype: float64


,Ticker,Reason
0,APP,Incomplete history (first price 2021-04-15)
1,COIN,Incomplete history (first price 2021-04-14)
2,CEG,Incomplete history (first price 2022-01-19)
3,EXE,Incomplete history (first price 2021-02-10)
4,FDXF,No data in study period
5,FERG,Zero or negative volume
6,GEHC,Incomplete history (first price 2022-12-15)
7,GEV,Incomplete history (first price 2024-03-27)
8,HONA,No data in study period
9,Q,No data in study period
